# ActiveLearningAgent — Experiment (Assignment 4)

Трек A: Active Learning.

Требования:
- AL-цикл: старт с N=50, 5 итераций по 20 примеров
- Сравнение стратегий: entropy vs random (можно добавить margin)
- Learning curves на одном графике
- Вывод: сколько примеров сэкономлено при том же качестве vs random baseline


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from al_agent import ActiveLearningAgent

data_dir = Path('data/raw')
candidates = [
    data_dir / 'merged_with_appendix.parquet',
    data_dir / 'merged_raw.parquet',
    data_dir / 'merged_raw.csv'
]
path = next((p for p in candidates if p.exists()), None)
assert path is not None, f'No dataset found. Looked for: {candidates}'
print('Loading:', path)

df = pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_csv(path)
df.head()

## Подготовка `label` для AL симуляции

AL требует, чтобы у pool были «oracle» метки (в реальной жизни их бы ставил человек).
Здесь мы строим метку из `seed_role` (лежит в `meta.seed_role` или уже распакован в отдельную колонку в других шагах).


In [ ]:
def parse_meta(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return {}
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return {'raw_meta': x}
    return {'raw_meta': str(x)}

def get_seed_role(row):
    if 'seed_role' in row and isinstance(row['seed_role'], str) and row['seed_role']:
        return row['seed_role']
    m = parse_meta(row.get('meta'))
    return m.get('seed_role')

df['seed_role'] = df.apply(get_seed_role, axis=1)
df[['seed_role','source','language']].head(10)

In [ ]:
# Маппинг seed_role -> oracle label (3-class)
def map_role_to_label(role: str) -> str:
    role = (role or '').strip()
    if role in {'native_ru_seed', 'candidate_benign_borderline'}:
        return 'candidate_benign_borderline'
    if role in {'safe_seed_en'}:
        return 'plain_benign_control'
    if role in {'unsafe_donor_ru', 'unsafe_donor'}:
        return 'unsafe_or_not_suitable'
    # fallback
    return 'plain_benign_control'

df['label'] = df['seed_role'].apply(map_role_to_label)

# оставим только непустые тексты
df['text'] = df['text'].astype('string').fillna('').str.strip()
df = df[df['text'] != ''].copy()

df['label'].value_counts()

## Split: test / train_pool, затем стартовые 50

Мы делаем один фиксированный test set, одинаковый для всех стратегий.


In [ ]:
train_pool, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# стартовые 50 (стратифицированно)
labeled_df, pool_df = train_test_split(
    train_pool,
    train_size=50,
    random_state=42,
    stratify=train_pool['label']
)

print('labeled:', labeled_df.shape, 'pool:', pool_df.shape, 'test:', test_df.shape)
print('labeled label dist:', labeled_df['label'].value_counts().to_dict())

## Запуск AL-цикла: entropy vs random


In [ ]:
agent = ActiveLearningAgent(model='logreg')

history_entropy = agent.run_cycle(
    labeled_df=labeled_df,
    pool_df=pool_df,
    test_df=test_df,
    strategy='entropy',
    n_iterations=5,
    batch_size=20
)

history_random = agent.run_cycle(
    labeled_df=labeled_df,
    pool_df=pool_df,
    test_df=test_df,
    strategy='random',
    n_iterations=5,
    batch_size=20
)

history_entropy[:2], history_random[:2]

In [ ]:
# График learning curves на одном графике
out = agent.report(
    {'entropy': history_entropy, 'random': history_random},
    metric='f1',
    output_path='reports/learning_curve.png',
    title='Active Learning: entropy vs random (macro F1)'
)
out

## Сколько примеров сэкономлено?

Сравним: какая минимальная разметка нужна стратегии entropy, чтобы достичь финального качества random.


In [ ]:
target_f1 = history_random[-1]['f1']
target_acc = history_random[-1]['accuracy']
print('Random final f1:', target_f1, 'acc:', target_acc)

best_n = None
for r in history_entropy:
    if r['f1'] >= target_f1:
        best_n = r['n_labeled']
        break

if best_n is None:
    print('Entropy did not reach random final F1 within 5 iterations.')
else:
    random_final_n = history_random[-1]['n_labeled']
    print('Entropy reaches random F1 at n_labeled =', best_n)
    print('Label savings vs random:', random_final_n - best_n)


## (Опционально) Сравнение с margin

Если хочешь — раскомментируй следующий блок.


In [ ]:
# history_margin = agent.run_cycle(
#     labeled_df=labeled_df,
#     pool_df=pool_df,
#     test_df=test_df,
#     strategy='margin',
#     n_iterations=5,
#     batch_size=20
# )
# agent.report({'entropy': history_entropy, 'random': history_random, 'margin': history_margin}, metric='f1',
#             output_path='reports/learning_curve_3ways.png', title='AL: entropy vs random vs margin')
